# Frame: tile index, blocks, grids, windows

Everything here is synthetic: a two-block tile index of 3200 m tiles in the equal-area frame.
See `docs/frame.md` for the model and `docs/conventions.md` for the pixel rules.

In [ ]:
import geopandas as gpd
from shapely.geometry import box

from idd_spatial_analysis.constants import WGS84, WORLD_CYLINDRICAL_EQUAL_AREA
from idd_spatial_analysis.frame import (
    AntimeridianError,
    bbox_in_crs,
    block_bounds,
    block_for_point,
    block_grid,
    blocks_intersecting,
    grid_from_bounds,
)

In [ ]:
# A tile index: one row per tile, a block_key per block, polygons in ESRI:54034.
rows = []
for bx, block_key in enumerate(("B-0000X-0000Y", "B-0001X-0000Y")):
    for tx in range(2):
        for ty in range(2):
            left, bottom = bx * 6400.0 + tx * 3200.0, ty * 3200.0
            rows.append(
                {
                    "block_key": block_key,
                    "geometry": box(left, bottom, left + 3200.0, bottom + 3200.0),
                }
            )
tiles = gpd.GeoDataFrame(rows, geometry="geometry", crs=WORLD_CYLINDRICAL_EQUAL_AREA)
tiles

In [ ]:
# Which blocks does a study area touch? Polygons in another CRS are reprojected.
study_area = gpd.GeoDataFrame(
    geometry=[box(6000, 1000, 8000, 2000)], crs=WORLD_CYLINDRICAL_EQUAL_AREA
)
print(blocks_intersecting(tiles, study_area))
print(blocks_intersecting(tiles, study_area.to_crs(WGS84)))
print(block_for_point(tiles, 7000.0, 100.0, WORLD_CYLINDRICAL_EQUAL_AREA))

In [ ]:
# The grid of one block at 100 m: anchored at the block's top-left corner.
grid = block_grid(tiles, "B-0001X-0000Y", resolution=100.0)
print(grid.shape, grid.resolution, grid.bounds)
print(block_bounds(tiles, "B-0001X-0000Y"))

In [ ]:
# Windows cover bounds with whole pixels; sub-grids keep the lattice.
window = grid.window((7000.0, 1000.0, 7550.0, 1300.0), pad=1)
sub = grid.sub(window)
print(window)
print(sub.shape, sub.bounds)
xs, ys = sub.pixel_centers()
print(xs[:3], ys[:3])

In [ ]:
# Bounds across CRSs: densified edges, and a guard against boxes that wrap the antimeridian.
print(grid.lonlat_bounds(pad=500.0))
print(bbox_in_crs((0.0, 0.0, 1.0, 1.0), WGS84, WORLD_CYLINDRICAL_EQUAL_AREA))
try:
    bbox_in_crs((-20_000_000.0, 0.0, 20_000_000.0, 100_000.0), WORLD_CYLINDRICAL_EQUAL_AREA, WGS84)
except AntimeridianError as exc:
    print("refused:", exc)

In [ ]:
# A grid for arbitrary bounds rounds the pixel count up, never the resolution.
small = grid_from_bounds((0.0, 0.0, 250.0, 199.9999999), 100.0, WORLD_CYLINDRICAL_EQUAL_AREA)
print(small.shape, small.bounds)